<a href="https://colab.research.google.com/github/venkataratnamb20/iiitk-fdp-agents-in-action/blob/main/notebooks/session02_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# langchain agents


<center>
<img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/core_agent_loop.svg?w=1650&fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=cb329408a32e8d35223b7a33bc407785" alt="Agent graph with model, tools and result" width="50%" height="50%"/>
</center>

## Topics
1. System Prompt
2. Tools
3. Structured Response
4. Memory
5. subagents
6. Human-In-The-Loop
7. Middlewares

## Setup

In [2]:
!pip install -Uq langchain-openai langchain-tavily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 5.6 MB/s eta 0:00:00


In [3]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

## First Agent

In [31]:
from langchain.agents import create_agent

# create agent
agent = create_agent(
    model='openai:gpt-4.1',
    tools=[],
    system_prompt = "You are a helpful assistant",
)

response = agent.invoke({'messages': ['Write python code for prime number']})

response

{'messages': [HumanMessage(content='Write python code for prime number', additional_kwargs={}, response_metadata={}, id='6708f9a3-1dc0-4500-9a48-c7c76a9adeb5'),
  AIMessage(content='Certainly! Here’s a simple Python function to check if a number is **prime**:\n\n```python\ndef is_prime(n):\n    if n <= 1:\n        return False\n    if n == 2:\n        return True\n    if n % 2 == 0:\n        return False\n    for i in range(3, int(n**0.5) + 1, 2):\n        if n % i == 0:\n            return False\n    return True\n\n# Example usage:\nnumber = 29\nif is_prime(number):\n    print(f"{number} is a prime number.")\nelse:\n    print(f"{number} is not a prime number.")\n```\n\n**Explanation:**  \n- Numbers less than or equal to 1 are not prime.\n- 2 is prime.\n- Even numbers greater than 2 are not prime.\n- The function checks divisibility **up to the square root** of `n` for efficiency.\n\n**Try replacing `number` with other values to test!**', additional_kwargs={'refusal': None}, response_m

**Understanding repsonse**

1) response is a dictionary with 'messages' as a key. Hence get `response['messages']`
2)  `response['messages']` is a python list. Last message is the recent response. Hence the response to the recent query is  `response['messages'][-1].content`

In [32]:
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Write python code for prime number
================================== Ai Message ==================================

Certainly! Here’s a simple Python function to check if a number is **prime**:

```python
def is_prime(n):
    if n <= 1:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return False
    return True

# Example usage:
number = 29
if is_prime(number):
    print(f"{number} is a prime number.")
else:
    print(f"{number} is not a prime number.")
```

**Explanation:**  
- Numbers less than or equal to 1 are not prime.
- 2 is prime.
- Even numbers greater than 2 are not prime.
- The function checks divisibility **up to the square root** of `n` for efficiency.

**Try replacing `number` with other values to test!**


In [42]:
import gradio as gr

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

######################################################
# agent
######################################################
def run_agent(
        message, history=[],
        model = 'openai:gpt-4.1',
        temperature = 0.7,
        max_tokens = 1000,
        system_prompt = "You are a helpful assistant"
):
    # llm

    llm = init_chat_model(
        model=model,
        temperature = temperature,
        max_tokens = max_tokens
    )
    # create agent
    agent = create_agent(
        # model='openai:gpt-4.1',
        model = llm,
        tools=[],
        system_prompt = system_prompt,
    )

    response = agent.invoke({'messages': [message]})
    return response['messages'][-1].content

######################################################
# openai models
######################################################
openai_models = [
    'openai:gpt-5.5',
    'openai:gpt-5',
    'openai:gpt-4.1',
    'openai:gpt-4o-mini',
    'openai:gpt-3.5-turbo',
]


######################################################
# GUI: Gradio
######################################################
with gr.Blocks() as app:
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            model = gr.Dropdown(
                label='Model',
                choices=openai_models,
                value='openai:gpt-4.1',
            )
            temperature = gr.Number(
                label='Temperature',
                value=0.7,
                minimum=0.0,
                maximum=1.0,
                step=0.01,
            )
            max_tokens = gr.Number(
                label='Max Tokens',
                value=1000,
                minimum=0,
                maximum=10000,
                step=1,
            )
            sys_prompt = gr.Textbox(
                label='System Prompt',
                value="",
                lines=5,
                placeholder="You are a helpful assistant",
            )
        with gr.Column(scale=3):
            chat_ui = gr.ChatInterface(
                run_agent,
                additional_inputs = [
                    model,
                    temperature,
                    max_tokens,
                    sys_prompt
                ],
                title = 'Agent Chat Bot',
                description = "Chat with agent",
                show_progress = 'minimal'
            )

# launch app
app.launch(theme='soft', debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://665a5e96253516c7cb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Tools and Structured Response

In [48]:
from pathlib import Path

from pydantic import BaseModel, Field

from langchain.agents import create_agent

from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

from langchain.tools import tool

# structured response
class Report(BaseModel):
    title: str = Field(description="The title of the report")
    summary: str = Field(description="A summary of the report")
    sources: list[str] = Field(description="A list of sources used to answer the question")

# tools
@tool
def write_file(content: str, filename: str='') -> str:
    """Write content to a file.

    if filename is not provided, it will be saved to 'content.md'
    if prent dirs doesn't exist, it will be created.

    Args:
        content (str): The content to write to the file.
        filename (str, optional): The name of the file to write to. Defaults to ''.

    Returns:
        str: A message indicating whether the file was written successfully or not.

    """

    if not filename:
        filename = 'content.md'
    filename = Path(filename)
    # create parent dirs if doesn't exist
    filename.parent.mkdir(parents=True, exist_ok=True)
    try:
        with open(filename, 'w') as f:
            f.write(content)
        return f'FileWriteSuccess: {filename}'
    except Exception as e:
        return f'FileWriteError: {filename}\n{e}'

# wweb search
web_search = TavilySearch()

# tools
tools = [
    web_search,
    write_file
]

# create agent
agent = create_agent(
    model='openai:gpt-4.1',
    tools=tools,
    system_prompt = "You are a helpful assistant",
    response_format = Report,
)

# message
message = 'Do a research and Write a detailed report about the "agentic and deepagents SDKs. Get metrics to compare all SDKs and create sections accordingly" and write in "./docs/report.md"'
result = agent.invoke({'messages': [message]})
result

{'messages': [HumanMessage(content='Do a research and Write a detailed report about the "agentic and deepagents SDKs. Get metrics to compare all SDKs and create sections accordingly" and write in "./docs/report.md"', additional_kwargs={}, response_metadata={}, id='b2750d20-a8c5-417e-841c-4ae4496d3568'),
  AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 323, 'prompt_tokens': 1503, 'total_tokens': 1826, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 86, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-EHeQyDhRtr5MEaMJEKyKaFGU0dM1B', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b2-5d65-7392-8beb-07e0d67c9c14-0', tool_ca

In [50]:
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

Do a research and Write a detailed report about the "agentic and deepagents SDKs. Get metrics to compare all SDKs and create sections accordingly" and write in "./docs/report.md"
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_kdnKiDg4a8zOJW0dJz57NW0c)
 Call ID: call_kdnKiDg4a8zOJW0dJz57NW0c
  Args:
    query: Agentic SDK GitHub agentic sdk TypeScript Python AI agents
    include_domains: None
    exclude_domains: None
    search_depth: advanced
    include_images: False
    time_range: None
    topic: general
    start_date: None
    end_date: None
  tavily_search (call_ihgDxPO62MOjINvzAAbnXHZk)
 Call ID: call_ihgDxPO62MOjINvzAAbnXHZk
  Args:
    query: DeepAgents SDK GitHub LangChain deepagents SDK documentation
    include_domains: None
    exclude_domains: None
    search_depth: advanced
    include_images: False
    time_range: None
 

In [51]:
result['structured_response']

Report(title='Research report written', summary='A detailed research report comparing Agentic and DeepAgents SDKs has been created at `./docs/report.md`. It includes SDK profiles, adoption/package metrics, capability matrices, qualitative scoring, security/governance notes, use-case fit, decision guidance, combination patterns, risks, and source links.', sources=['https://github.com/transitive-bullshit/agentic', 'https://www.npmjs.com/package/@agentic/core', 'https://www.npmjs.com/package/@agentic/stdlib', 'https://github.com/langchain-ai/deepagents', 'https://github.com/langchain-ai/deepagents/releases', 'https://pypi.org/project/deepagents', 'https://docs.langchain.com/oss/python/deepagents/overview', 'https://reference.langchain.com/python/deepagents/graph/create_deep_agent', 'https://reference.langchain.com/javascript/deepagents', 'https://github.com/langchain-ai/managed-deepagents'])

In [52]:
def display_report(report: Report):
    from IPython.display import display, Markdown

    sources = '\n'+'\n'.join([f'{idx+1}. {res}' for idx, res in enumerate(report.sources)])
    content = f"""
    # {report.title}

    ## Summary

    {report.summary}

    ## Sources

    {sources}
    """
    content = content.replace("    ", "")
    display(Markdown(content))

display_report(result['structured_response'])


# Research report written

## Summary

A detailed research report comparing Agentic and DeepAgents SDKs has been created at `./docs/report.md`. It includes SDK profiles, adoption/package metrics, capability matrices, qualitative scoring, security/governance notes, use-case fit, decision guidance, combination patterns, risks, and source links.

## Sources


1. https://github.com/transitive-bullshit/agentic
2. https://www.npmjs.com/package/@agentic/core
3. https://www.npmjs.com/package/@agentic/stdlib
4. https://github.com/langchain-ai/deepagents
5. https://github.com/langchain-ai/deepagents/releases
6. https://pypi.org/project/deepagents
7. https://docs.langchain.com/oss/python/deepagents/overview
8. https://reference.langchain.com/python/deepagents/graph/create_deep_agent
9. https://reference.langchain.com/javascript/deepagents
10. https://github.com/langchain-ai/managed-deepagents


## Agent Cht Bot with Tools

In [60]:
import os
import subprocess
import shutil

from uuid import uuid4

import requests

import gradio as gr

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

from pathlib import Path

from pydantic import BaseModel, Field

from langchain.agents import create_agent

from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

from langchain.tools import tool

# structured response
class Report(BaseModel):
    title: str = Field(description="The title of the report")
    summary: str = Field(description="A summary of the report")
    sources: list[str] = Field(description="A list of sources used to answer the question")

# tools
@tool
def write_file(content: str, filename: str='') -> str:
    """Write content to a file.

    if filename is not provided, it will be saved to 'content.md'
    if prent dirs doesn't exist, it will be created.

    Args:
        content (str): The content to write to the file.
        filename (str, optional): The name of the file to write to. Defaults to ''.

    Returns:
        str: A message indicating whether the file was written successfully or not.

    """

    if not filename:
        filename = 'content.md'
    filename = Path(filename)
    # create parent dirs if doesn't exist
    filename.parent.mkdir(parents=True, exist_ok=True)
    try:
        with open(filename, 'w') as f:
            f.write(content)
        return f'FileWriteSuccess: {filename}'
    except Exception as e:
        return f'FileWriteError: {filename}\n{e}'

@tool
def read_file(filename: str) -> str:
    """Read content from a file.

    Args:
        filename (str): The name of the file to read from.

    Returns:
        str: The content of the file.

    """
    try:
        with open(filename, 'r') as f:
            return f.read()
    except Exception as e:
        return f'FileReadError: {filename}\n{e}'

def makedir(dirname: str) -> str:
    """Create a directory.

    Args:
        dirname (str): The name of the directory to create.

    Returns:
        str: A message indicating whether the directory was created successfully or not.
    """
    _dirname = Path(dirname)
    try:
        _dirname.mkdir(parents=True, exist_ok=True)
        return f'DirCreateSuccess: {_dirname}'
    except Exception as e:
        return f'DirCreateError: {_dirname}\n{e}'

def download_pdf(url: str, filename: str='') -> str:
    """Download a PDF file from a URL.

    if filename is not provided, it will be saved to 'content.md'
    if prent dirs doesn't exist, it will be created.

    Args:
        url (str): The URL of the PDF file to download.
        filename (str, optional): The name of the file to save the PDF to. Defaults to ''.

    Returns:
        str: A message indicating whether the file was downloaded successfully or not.
    """
    if not filename:
        filename = url.split('/')[-1].strip().split('?')[0]
    if not filename.endswith('.pdf'):
        filename += '.pdf'
    _filename = Path(filename)
    # create parent dirs
    _filename.parent.mkdir(parents=True, exist_ok=True)
    try:
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
            return f'FileDownloadSuccess: {_filename}'
    except Exception as e:
        return f'FileDownloadError: {_filename}\n{e}'

def download_image(url: str, filename: str='') -> str:
    """Download image from a URL.

    if filename is not provided, it will be saved to 'content.md'
    if prent dirs doesn't exist, it will be created.

    Args:
        url (str): The URL of the image file to download.
        filename (str, optional): The name of the file to save the image to. Defaults to ''.

    Returns:
        str: A message indicating whether the file was downloaded successfully or not.
    """
    image_extensions = [
        'JPG',
        'JPEG',
        'PNG',
        'SVG',
        'GIF',
        'BMP',
        'TIFF',
        'WEBP',
    ]
    if not filename:
        filename = url.split('/')[-1].strip().split('?')[0]
    file_extension = os.path.splitext(filename)[1]
    if file_extension.strip('.').lower() not in image_extensions.lower():
        filename += '.png'
    _filename = Path(filename)
    # create parent dirs
    _filename.parent.mkdir(parents=True, exist_ok=True)
    try:
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
            return f'FileDownloadSuccess: {_filename}'
    except Exception as e:
        return f'FileDownloadError: {_filename}\n{e}'

def run_shell(command: str) -> str:
    """Run a shell command and return the output.

    Args:
        command (str): The shell command to run.

    Returns:
        str: The output of the shell command.
    """
    try:
        output = subprocess.check_output(command, shell=True, stderr=subprocess.STDOUT)
        return f'ShellOutput: {output}'
    except Exception as e:
        return f'ShellError: {e}'

def run_python(code: str) -> str:
    """Run python code and return the output.

    Args:
        code (str): The python code to run.

    Returns:
        str: The output of the python code.
    """
    try:
        output = eval(code)
        return f'PythonOutput: {output}'
    except Exception as e:
        return f'PythonError: {e}'

def run_python_with_shell(code: str) -> str:
    """Run python code with shell and return the output.

    Args:
        code (str): The python code to run.

    Returns:
        str: The output of the python code.
    """
    result = run_shell(f'python -c {code}')

# https://myrightbird.com/assets/uploads/mybird_sun_conure_on_perch.jpg

# wweb search
web_search = TavilySearch()

# tools
tools = [
    web_search,
    write_file,
    read_file,
    makedir,
    download_image,
    download_pdf,
    run_shell,
    run_python_with_shell
]

# memory
memory = InMemorySaver()
config = {'configurable': {'thread_id': uuid4().hex}}

# create agent
######################################################
# agent
######################################################
def run_agent(
        message, history=[],
        model = 'openai:gpt-4.1',
        temperature = 0.7,
        max_tokens = 1000,
        system_prompt = "You are a helpful assistant"
):
    # llm

    llm = init_chat_model(
        model=model,
        temperature = temperature,
        max_tokens = max_tokens
    )
    # create agent
    agent = create_agent(
        # model='openai:gpt-4.1',
        model = llm,
        tools=tools,
        system_prompt = system_prompt,
        checkpointer = memory
    )

    response = agent.invoke(
        {'messages': [message]},
        config = config
        )
    return response['messages'][-1].content

######################################################
# openai models
######################################################
openai_models = [
    'openai:gpt-5.6',
    'openai:gpt-5.6-luna',
    'openai:gpt-5.6-sol',
    'openai:gpt-5.5',
    'openai:gpt-5.4-nano',
    'openai:gpt-5.4-mini',
    'openai:gpt-5.3-codex',
    # 'openai:gpt-5.1-codex-mini',
    'openai:gpt-5',
    'openai:gpt-5-nano',
    'openai:gpt-5-mini',
    'openai:gpt-4.1',
    'openai:gpt-4o-mini',
    'openai:gpt-3.5-turbo',
]

sorted_openai_models = sorted(openai_models)

######################################################
# GUI: Gradio
######################################################
with gr.Blocks() as app:
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            model = gr.Dropdown(
                label='Model',
                choices=sorted_openai_models,
                value='openai:gpt-4.1',
            )
            temperature = gr.Number(
                label='Temperature',
                value=0.7,
                minimum=0.0,
                maximum=1.0,
                step=0.01,
            )
            max_tokens = gr.Number(
                label='Max Tokens',
                value=1000,
                minimum=0,
                maximum=10000,
                step=1,
            )
            sys_prompt = gr.Textbox(
                label='System Prompt',
                value="",
                lines=5,
                placeholder="You are a helpful assistant",
            )
        with gr.Column(scale=3):
            chat_ui = gr.ChatInterface(
                run_agent,
                additional_inputs = [
                    model,
                    temperature,
                    max_tokens,
                    sys_prompt
                ],
                title = 'Agent Chat Bot',
                description = "Chat with agent",
                show_progress = 'minimal'
            )

# launch app
app.launch(theme='soft', debug=True)
# https://developers.openai.com/api/docs/models

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5301e1c0feb431537a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7873 <> https://5301e1c0feb431537a.gradio.live


## Prompt

```
Consider you are a senior software developer and python expert.

You are tasked to discuss with user and give software solutions as per user request.

## Guideliens
1. take tiem to understand user requirements. Ask questions if you need more information.
2.  Python development guideliens
- Follow CLEAN code principles. Write necessary comments and use google doc string format.
- Follow TDD.
3. Think step by step. Divide tasks into subtasks. solve each subtask and combine all results to give final solution,

## Development Guidelines
1) Plan to Make a list of TODOs. Write in the project directory and plan to develop according to the list of TODOs and keep track of the progress.
2) Write, run, debug, fix and iterate until the program works according to the user requirement.
``

## Memory

In [25]:
from uuid import uuid4

from pydantic import BaseModel, Field

from langchain.agents import create_agent

from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# structured response
class Report(BaseModel):
    title: str = Field(description="The title of the report")
    summary: str = Field(description="A summary of the report")
    sources: list[str] = Field(description="A list of sources used to answer the question")

# tools
web_search = TavilySearch()
tools = [
    web_search
]

# memory
memory = InMemorySaver()

# create agent
agent = create_agent(
    model='openai:gpt-4.1',
    tools=tools,
    system_prompt = "You are a helpful assistant",
    response_format = Report,
    checkpointer = memory
)

# add config in invoke if checkpointer or memory is added
config = {'configurable': {'thread_id': uuid4().hex}}

# message
message = 'Do a research and Write a detailed report about the "agentic and deepagents SDKs. Get metrics to compare all SDKs and create sections accordingly"'

# practice to add message in AIMessage format
result = agent.invoke(
    {'messages': [{'role': 'user', 'content': message}]},
    config=config
)
result

{'messages': [HumanMessage(content='Do a research and Write a detailed report about the "agentic and deepagents SDKs. Get metrics to compare all SDKs and create sections accordingly"', additional_kwargs={}, response_metadata={}, id='9cf713ba-3f88-465e-bbf5-4da675e76a6b'),
  AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 212, 'prompt_tokens': 1307, 'total_tokens': 1519, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_a728e95d1d', 'id': 'chatcmpl-EHdRO8d5NK5h18zo4C08xdo1wCMou', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04578-1874-7302-bc13-5abb1039afff-0', tool_calls=[{'name': 'tavily_

In [24]:
from uuid import uuid4

uuid4().hex[:6]

'b3bc31'

In [28]:

display_report(result['structured_response'])


# Detailed Report: Agentic vs Deepagents SDKs — Features, Architecture, and Benchmark Metrics

## Summary

This report presents a comprehensive comparison between two leading SDKs for agentic AI development: Agentic SDK (focusing on OpenAI Agents SDK and related tools) and Deepagents SDK (LangChain’s Deep Agents harness). The structure covers:

1. **Overview and Architecture:**
   - Agentic SDK is designed around minimal primitives (agents, handoffs, guardrails) for fast, simple, GPT-centric agent development.
   - Deepagents SDK, built on LangChain and LangGraph, offers a batteries-included experience with sub-agents, task planning, human-in-the-loop features, persistent memory, and virtual filesystem context management.

2. **Feature Comparison:**
   - Agentic SDK is lauded for rapid prototyping, consistent APIs (Python/TypeScript), and integrated guardrails for low-friction, production-ready deployments with OpenAI infrastructure.
   - Deepagents SDK excels for complex, stateful, multi-step agent workflows, supporting planning, sub-agent spawning, and rich context management (e.g., through virtual filesystems and sandboxed execution).
   - Both support Model Context Protocol (MCP) tools, enabling plug-and-play extensions, but Deepagents provides more granular control over agent orchestration, tool use, and memory.

3. **Key Benchmark and Evaluation Metrics:**
   - Industry-standard metrics to compare AI agent SDKs include: task completion and success rates; process metrics (tool selection/invocation, intent resolution, response relevance); cost-normalized accuracy; policy/safety adherence; latency and scalability; and observability (tracing, debugging, logging capabilities).
   - LangGraph/Deepagents offer deeper stateful execution and context management (giving advantages in long-horizon/complex tasks), while Agentic SDKs standardize on simplicity and ease of deployment (making them compelling for iterative development and fast launches).

4. **Performance Insights:**
   - Benchmarking shows framework choice alone can cause up to 30% difference in agent performance on identical LLMs ([uvik.net](https://uvik.net/blog/agentic-ai-frameworks)).
   - Deepagents shines in multi-agent and research/coding scenarios where planning, persistent memory, and delegated sub-agents are essential.
   - Agentic SDKs (notably OpenAI’s) provide the shortest path to production for tasks that do not require extensive agentic planning or context management.

5. **Ecosystem and Developer Experience:**
   - Agentic SDK (OpenAI) offers low barrier to entry, favorable for rapid development cycles, and robust integration with OpenAI APIs.
   - Deepagents SDK leverages the broader LangChain ecosystem, providing extensibility, stateful orchestration, and fine-tuned control via LangGraph and rich sandbox/tools support.
   - Tooling (observability, evaluation harnesses) is maturing, with new standards (CLEAR, GAIA2, tau2-bench) being prominent in agent benchmarking.

**Summary Table:**

| Metric| Agentic SDK (OpenAI, Claude, etc)   | Deepagents SDK (LangChain)  |
|-------------------------------|-----------------------------------------------------|-------------------------------------|
| Architecture/Approach | Lightweight primitives, model-centric| Batteries-included, planner+subagent|
| Context Management| Implicit (schema/guardrails)| Explicit (virtual filesystem, memory)|
| Tool/API Ecosystem| MCP, OpenAI tools, vendor lock-in possible  | MCP, custom/community tools |
| Planning/Subtasks | Minimal, relies on agent handoffs   | Native, via subagents/planning  |
| Observability/Tracing | Good (standardized tracing, logs)   | Excellent (graph-based tracing, logs)|
| Scalability/Statefulness  | Moderate (per-call context) | High (durable sessions, recall) |
| Developer Experience  | Fast, low friction, clear onboarding| More complex, greater flexibility   |
| Best Use Cases| Rapid prototyping, GPT-only workflows   | Multi-step, research, coding agents |

**Conclusion:**
- Choose Agentic SDKs (e.g., OpenAI) when fast iteration, OpenAI-centric workflows, and ease of use are top priorities.
- Opt for Deepagents SDK for complex, stateful, multi-step agent applications where extensibility, fine-grained control, and persistent context are critical.

Both ecosystems are advancing rapidly; benchmarking and observability should steer production architecture and performance evaluation.

## Sources


1. https://github.com/promptfoo/promptfoo/tree/main/examples/agentic-sdk-comparison
2. https://docs.langchain.com/oss/python/deepagents/overview
3. https://dev.to/thedailyagent/langchain-deep-agents-vs-openai-agents-sdk-2026-2bb1
4. https://reference.langchain.com/python/deepagents
5. https://uvik.net/blog/agentic-ai-frameworks
6. https://kili-technology.com/blog/agentic-ai-benchmarks-guide-what-they-are-how-they-work
7. https://langfuse.com/blog/2025-03-19-ai-agent-comparison
8. https://mem0.ai/blog/openai-agents-sdk-review
9. https://www.morphllm.com/ai-agent-framework
10. https://talkpython.fm/episodes/show/543/deep-agents-langchains-sdk-for-agents-that-plan-and-delegate
